In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()
        # self.activation = nn.Tanh()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
# X, y = make_friedman1(n_samples=10, noise=1e-2)
# X, y = make_friedman1(n_samples=100, noise=1e-2)
# X, y = make_friedman1(n_samples=1000, noise=1e-2)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonRaphson(model=model)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.20448531210422516
epoch:  1, loss: 0.16197150945663452
epoch:  2, loss: 0.11755184829235077
epoch:  3, loss: 0.1002705916762352
epoch:  4, loss: 0.08191826939582825
epoch:  5, loss: 0.06914496421813965
epoch:  6, loss: 0.05824736878275871
epoch:  7, loss: 0.026766648516058922
epoch:  8, loss: 0.007016324903815985
epoch:  9, loss: 0.006702047307044268
epoch:  10, loss: 0.006385363172739744
epoch:  11, loss: 0.006095515564084053
epoch:  12, loss: 0.00581772206351161
epoch:  13, loss: 0.005582321900874376
epoch:  14, loss: 0.004713419359177351
epoch:  15, loss: 0.0044048987329006195
epoch:  16, loss: 0.0027087873313575983
epoch:  17, loss: 0.002249474637210369
epoch:  18, loss: 0.0019068643450737
epoch:  19, loss: 0.0018731191521510482
epoch:  20, loss: 0.00184152415022254
epoch:  21, loss: 0.0015734719345346093
epoch:  22, loss: 0.0013968116836622357
epoch:  23, loss: 0.0012369209434837103
epoch:  24, loss: 0.0011084919096902013
epoch:  25, loss: 0.0010078439954668283


In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9961995482444763
Test metrics:  R2 = 0.9634224772453308


In [ ]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonRaphson(model=model, batch_size=100)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.18459662795066833
epoch:  1, loss: 0.18119099736213684
epoch:  2, loss: 0.15876062214374542
epoch:  3, loss: 0.12900099158287048
epoch:  4, loss: 0.10200416296720505
epoch:  5, loss: 0.09435034543275833
epoch:  6, loss: 0.07910002768039703
epoch:  7, loss: 0.06646546721458435
epoch:  8, loss: 0.03300342336297035
epoch:  9, loss: 0.011614974588155746
epoch:  10, loss: 0.006037616170942783
epoch:  11, loss: 0.003659293055534363
epoch:  12, loss: 0.0026703656185418367
epoch:  13, loss: 0.0020336469169706106
epoch:  14, loss: 0.0016482856590300798
epoch:  15, loss: 0.0013881188351660967
epoch:  16, loss: 0.0012020786525681615
epoch:  17, loss: 0.001063219620846212
epoch:  18, loss: 0.000957489712163806
epoch:  19, loss: 0.0008716921438463032
epoch:  20, loss: 0.0007959490758366883
epoch:  21, loss: 0.0007353649125434458
epoch:  22, loss: 0.0006871455116197467
epoch:  23, loss: 0.0006427709595300257
epoch:  24, loss: 0.0006087225046940148
epoch:  25, loss: 0.0005783017841

In [10]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9959132075309753
Test metrics:  R2 = 0.9264435172080994
